# Séance 9 — Projet du groupe 1 : activité d’une bibliothèque

Le canevas est rempli avec les quatre fichiers attribués à
data/groupe_01_bibliotheque/groupe_01_bibliotheque. Les noms des membres et la date de
soutenance restent à renseigner.

## Identification

| Élément | Valeur |
|---|---|
| Groupe | Groupe 1 — bibliothèque |
| Membres | À renseigner |
| Sujet | Activité d’emprunt, retards et pertes par site et famille d’ouvrage |
| Question | Quels sites et familles concentrent l’activité, les retards et les pertes d’ouvrages ? |
| Date de soutenance | À renseigner |

Format prévu : 15 minutes de présentation, 10 minutes de questions.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

> **Votre jeu de données vous est fourni.** Chaque groupe reçoit un domaine différent,
> avec ses propres anomalies : les chiffres, les pièges et les décisions à justifier ne
> sont pas les mêmes d'un groupe à l'autre. Comparer vos *méthodes* entre groupes est
> encouragé ; comparer vos *résultats* ne vous mènera nulle part.

> **Aucun sujet ne dit quelles anomalies son jeu contient.** Les trouver est la première
> partie du travail, et la première chose que le jury vérifiera. Commencez par appliquer
> votre `profile_dataframe` du jour 1 : elle a été écrite pour cela.

## 1. Contexte (2 min de présentation)

Le jeu attribué au groupe 1 décrit des événements d’emprunt en bibliothèque et trois
référentiels : adhérents, ouvrages et sites. La table principale contient environ 20 500
événements. Les données couvrent plusieurs années et comportent des dates hétérogènes,
des valeurs manquantes, des statuts et modes de saisie dont la casse et les espaces varient,
ainsi que des identifiants qui ne se joignent pas tous aux référentiels.

La question métier retenue est : **quels sites et familles d’ouvrages concentrent l’activité,
les retards et les pertes, et comment ces indicateurs évoluent-ils par mois ?**

Un retard est un événement dont le statut normalisé vaut en_retard. La valeur perdue est
la valeur déclarée de l’ouvrage uniquement pour les événements au statut perdu. Cela mesure
une exposition déclarée, pas une perte comptable validée.

In [ ]:
# Chargement des sources réellement attribuées au groupe 1
GROUP_DIR = ROOT / 'data' / 'groupe_01_bibliotheque' / 'groupe_01_bibliotheque'
BORROWS_PATH = GROUP_DIR / 'emprunts.csv'
MEMBERS_PATH = GROUP_DIR / 'adherents.csv'
BOOKS_PATH = GROUP_DIR / 'ouvrages.csv'
SITES_PATH = GROUP_DIR / 'sites.csv'

for path in (BORROWS_PATH, MEMBERS_PATH, BOOKS_PATH, SITES_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Fichier attendu introuvable : {path}')
    print(f'{path.name:<20} {path.stat().st_size / 1024:>8.1f} Ko')

raw_borrows = pd.read_csv(BORROWS_PATH)
raw_members = pd.read_csv(MEMBERS_PATH)
raw_books = pd.read_csv(BOOKS_PATH)
raw_sites = pd.read_csv(SITES_PATH)

print('
Emprunts :', raw_borrows.shape)
display(raw_borrows.head())
print('Adhérents :', raw_members.shape, '| Ouvrages :', raw_books.shape,
      '| Sites :', raw_sites.shape)
display(raw_books.head(3))
display(raw_sites.head(3))

---
## 2. Diagnostic (3 min)

Appliquez votre fonction `profile_dataframe` de la séance 2, puis **quantifiez** chaque
anomalie. Un diagnostic sans chiffre n'est pas un diagnostic.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

from exploration import profile_dataframe

def normalize_library_text(series):
    text = series.astype('string').str.strip().str.lower()
    text = text.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('utf-8')
    return text.str.replace(r'\s+', ' ', regex=True)


DATE_FORMATS = ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y',
                '%Y/%m/%d %H:%M', '%d %b %Y']

def parse_library_dates(series):
    parsed = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    for date_format in DATE_FORMATS:
        parsed = parsed.fillna(
            pd.to_datetime(series, format=date_format, errors='coerce')
        )
    return parsed


profile_dataframe(raw_borrows, name='emprunts bruts', max_cardinality=15)

parsed_dates = parse_library_dates(raw_borrows['date_evenement'])
id_counts = raw_borrows['id_emprunt'].value_counts(dropna=True)
reused_id_rows = int(raw_borrows['id_emprunt'].isin(id_counts[id_counts > 1].index).sum())
missing_duration = pd.to_numeric(
    raw_borrows['duree_pret_jours'], errors='coerce'
).isna().sum()
missing_value = pd.to_numeric(
    raw_borrows['valeur_ouvrage_eur'].astype('string')
        .str.replace(r'(?i)\s*EUR$', '', regex=True)
        .str.replace(',', '.', regex=False),
    errors='coerce',
).isna().sum()
orphan_counts = {
    'id_adherent': int((~raw_borrows['id_adherent'].isin(raw_members['id_adherent'])).sum()),
    'id_ouvrage': int((~raw_borrows['id_ouvrage'].isin(raw_books['id_ouvrage'])).sum()),
    'id_site': int((~raw_borrows['id_site'].isin(raw_sites['id_site'])).sum()),
}

diagnostic = pd.DataFrame([
    {'indicateur': 'Lignes brutes', 'valeur': len(raw_borrows)},
    {'indicateur': 'Doublons stricts', 'valeur': int(raw_borrows.duplicated().sum())},
    {'indicateur': 'Lignes portant un id_emprunt réutilisé', 'valeur': reused_id_rows},
    {'indicateur': 'Date manquante ou non reconnue', 'valeur': int(parsed_dates.isna().sum())},
    {'indicateur': 'Durée manquante/non numérique', 'valeur': int(missing_duration)},
    {'indicateur': 'Valeur d’ouvrage non convertible', 'valeur': int(missing_value)},
    {'indicateur': 'Adhérent non référencé', 'valeur': orphan_counts['id_adherent']},
    {'indicateur': 'Ouvrage non référencé', 'valeur': orphan_counts['id_ouvrage']},
    {'indicateur': 'Site non référencé', 'valeur': orphan_counts['id_site']},
])
diagnostic['part_%'] = (diagnostic['valeur'] / len(raw_borrows) * 100).round(2)
diagnostic

### Tableau des anomalies

| Anomalie | Colonne | Ampleur observée | Gravité | Traitement retenu |
|---|---|---:|---|---|
| Dates absentes ou non reconnues | date_evenement | 412 / 20 563 (2,00 %) | Élevée pour l’analyse temporelle | Exclure de l’analyse après comptage |
| Durées manquantes | duree_pret_jours | 1 841 / 20 563 (8,96 %) | Moyenne | Conserver comme manquant ; ne pas inventer une durée |
| Identifiant d’emprunt réutilisé | id_emprunt | 180 occurrences au-delà de la première | À examiner | Ne pas dédoublonner sur l’identifiant seul ; vérifier les statuts |
| Casse et espaces instables | statut, mode_emprunt | Des modalités apparaissent avec espaces/casse différents | Moyenne | Normaliser avant les regroupements |
| Clés de faits sans référentiel | id_adherent, id_ouvrage, id_site | Comptées dans le diagnostic dynamique | Élevée pour les analyses ventilées | Jointures gauches et suivi des orphelins |

Les nombres de la première colonne correspondent au jeu attribué présent dans le dépôt.
La cellule de diagnostic recalcule les valeurs et les taux depuis les CSV.

---
## 3. Traitements (5 min — la partie la plus longue de la soutenance)

Pour chaque décision, trois éléments sont attendus : **ce que vous avez fait**, **pourquoi**,
et **quelle alternative vous avez écartée**. Le troisième point est celui qui fait la
différence à l'oral.

In [ ]:
# Pipeline Pandas de référence : normalisation, contrôle et enrichissement
borrows = raw_borrows.copy()

for column in ['mode_emprunt', 'statut']:
    borrows[column] = normalize_library_text(borrows[column])

borrows['date_evenement'] = parse_library_dates(borrows['date_evenement'])
borrows['duree_pret_jours'] = pd.to_numeric(
    borrows['duree_pret_jours'], errors='coerce'
)
borrows['nb_prolongations'] = pd.to_numeric(
    borrows['nb_prolongations'], errors='coerce'
)
borrows['valeur_ouvrage_eur'] = pd.to_numeric(
    borrows['valeur_ouvrage_eur'].astype('string')
        .str.replace(r'(?i)\s*EUR$', '', regex=True)
        .str.replace(',', '.', regex=False),
    errors='coerce',
)

exact_duplicates = int(borrows.duplicated().sum())
borrows = borrows.drop_duplicates()
invalid_dates = int(borrows['date_evenement'].isna().sum())
borrows = borrows.dropna(subset=['date_evenement'])
borrows = borrows[
    borrows['duree_pret_jours'].isna() | (borrows['duree_pret_jours'] >= 0)
].copy()
print('Doublons stricts supprimés :', exact_duplicates)
print('Dates manquantes/non reconnues exclues :', invalid_dates)

books = raw_books.copy()
for column in ['famille', 'sous_famille']:
    books[column] = normalize_library_text(books[column])
books['cout_reference_eur'] = pd.to_numeric(
    books['cout_reference_eur'], errors='coerce'
)
book_medians = books.groupby('sous_famille')['cout_reference_eur'].transform('median')
books['cout_reference_eur'] = books['cout_reference_eur'].fillna(book_medians)

sites = raw_sites.copy()
for column in ['region', 'ville']:
    sites[column] = normalize_library_text(sites[column])
members = raw_members.copy()
members['segment'] = normalize_library_text(members['segment'])
members['age'] = pd.to_numeric(members['age'], errors='coerce')

library_project = (borrows
    .merge(books, on='id_ouvrage', how='left', validate='many_to_one')
    .merge(sites, on='id_site', how='left', validate='many_to_one')
    .merge(members[['id_adherent', 'segment', 'age']],
           on='id_adherent', how='left', validate='many_to_one'))

age_medians = library_project.groupby('segment')['age'].transform('median')
library_project['age_impute'] = library_project['age'].fillna(age_medians)
library_project['annee'] = library_project['date_evenement'].dt.year
library_project['mois'] = library_project['date_evenement'].dt.to_period('M').astype(str)
library_project['est_en_retard'] = library_project['statut'].eq('en_retard').astype(int)
library_project['est_perdu'] = library_project['statut'].eq('perdu').astype(int)
library_project['valeur_perdue_eur'] = library_project[
    'valeur_ouvrage_eur'
].where(library_project['statut'].eq('perdu'), 0.0)

print('Emprunts finaux :', len(library_project))
print('Ouvrages non appariés :', int(library_project['famille'].isna().sum()))
print('Sites non appariés :', int(library_project['region'].isna().sum()))
print('Adhérents non appariés :', int(library_project['segment'].isna().sum()))
display(library_project.head())
save_dataset(library_project, 'bibliotheque_projet_pandas')

### Journal des décisions

| Décision | Justification | Alternative écartée | Motif du rejet |
|---|---|---|---|
| Essayer les formats de date explicitement | Une date invalide ne doit pas être devinée ni placer un événement au mauvais mois | Inférence de date automatique | Les dates ambiguës jour/mois peuvent être inversées |
| Normaliser espaces, casse et accents dans les champs descriptifs | Évite de créer plusieurs catégories pour une même modalité | Modifier aussi les identifiants | Les clés doivent rester strictement conformes aux référentiels |
| Supprimer les doublons stricts, pas les id_emprunt réutilisés | Un même identifiant peut porter plusieurs états d’un événement ; 180 occurrences supplémentaires doivent être inspectées | Garder uniquement la première ligne par identifiant | Cela masquerait des statuts ou corrections distincts |
| Conserver les durées manquantes et les exclure seulement des moyennes | Aucune durée ne peut être déduite des autres colonnes | Imputer une durée moyenne globale | Cela ferait paraître précis un délai inconnu |
| Garder les événements sans dimension correspondante avec une jointure gauche | Préserve le volume et rend visibles les clés orphelines | Jointure interne | Elle ferait disparaître silencieusement des emprunts |
| Utiliser un broadcast pour les petits référentiels | Les livres, sites et adhérents sont petits face aux événements | Trier/shuffler la table de faits pour chaque jointure | Coût inutile en local ; le plan Spark est contrôlé dans le notebook |
| Imputer l’âge manquant par médiane de segment | Limite l’effet des segments d’âges différents | Médiane globale ou suppression | La médiane globale ignore le segment ; supprimer réduit l’échantillon |

---
## 4. Composant PySpark (2 min)

Quelle étape avez-vous portée sur Spark, et **pourquoi celle-là** ? Chiffrez : volume
traité, durée, mémoire. Rappel de la séance 7 : « parce que c'est plus rapide » est une
réponse fausse sur une machine unique.

In [ ]:
# Étape PySpark et mesures réellement observées
from library_pipeline import create_spark_session, build_library_dataframe
import psutil
import time

spark = create_spark_session('PMD-groupe-01-mesure')
spark.sparkContext.setLogLevel('ERROR')
memory_before_mb = psutil.Process().memory_info().rss / 1024 ** 2
start = time.perf_counter()
library_spark = build_library_dataframe(spark, GROUP_DIR)
library_row_count = library_spark.count()
elapsed_seconds = time.perf_counter() - start
memory_after_mb = psutil.Process().memory_info().rss / 1024 ** 2

print(f'Événements finaux : {library_row_count:,}')
print(f'Durée de nettoyage et jointure : {elapsed_seconds:.2f} s')
print(f'RSS Python avant/après : {memory_before_mb:.1f} / {memory_after_mb:.1f} Mo')
print('La valeur RSS mesure le processus Python, pas toute la mémoire JVM de Spark.')
spark.stop()

---
## 5. Résultat (3 min)

Le jeu final, sa structure, les indicateurs, la planche de visualisation, et les limites
que vous assumez. **Énoncer soi-même les limites de son travail est valorisé** : cela
montre que vous savez où il ne faut pas faire confiance à vos données.

In [ ]:
# Jeu final PySpark : contrôles, agrégations et visualisations
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

OUTPUT_PATH = PROCESSED_DIR / 'bibliotheque_projet_spark'
spark = create_spark_session('PMD-groupe-01-resultat')
spark.sparkContext.setLogLevel('ERROR')

project_df = build_library_dataframe(spark, GROUP_DIR)
(project_df.write
    .mode('overwrite')
    .partitionBy('annee')
    .parquet(str(OUTPUT_PATH)))
final_library = spark.read.parquet(str(OUTPUT_PATH))

print(f'Jeu final : {final_library.count():,} lignes')
print('Colonnes :', final_library.columns)
for label, column in [
    ('Ouvrages non appariés', 'famille'),
    ('Sites non appariés', 'region'),
    ('Adhérents non appariés', 'segment'),
]:
    print(label + ':', final_library.filter(F.col(column).isNull()).count())

activity_by_site_family = (final_library
    .groupBy('region', 'famille')
    .agg(
        F.count('*').alias('emprunts'),
        F.sum('est_en_retard').alias('en_retard'),
        F.sum('est_perdu').alias('perdus'),
        F.round(F.avg('duree_pret_jours'), 1).alias('duree_moyenne_jours'),
        F.round(F.sum('valeur_perdue_eur'), 2).alias('valeur_perdue_eur'),
    )
    .withColumn(
        'taux_retard_pct',
        F.round(F.col('en_retard') / F.col('emprunts') * 100, 2),
    )
    .orderBy(F.desc('emprunts')))

monthly_activity = (final_library
    .groupBy('annee', 'mois')
    .agg(
        F.count('*').alias('emprunts'),
        F.sum('est_en_retard').alias('en_retard'),
        F.sum('est_perdu').alias('perdus'),
    )
    .orderBy('annee', 'mois'))

display(activity_by_site_family.toPandas())
activity_plot = monthly_activity.toPandas()
activity_plot['periode'] = (
    activity_plot['annee'].astype(str) + '-' +
    activity_plot['mois'].astype(str).str.zfill(2)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
site_plot = activity_by_site_family.toPandas().head(12).copy()
site_plot['site_famille'] = (
    site_plot['region'].fillna('non appariée') + ' / ' +
    site_plot['famille'].fillna('non appariée')
)
axes[0].barh(
    site_plot['site_famille'][::-1],
    site_plot['emprunts'][::-1],
)
axes[0].set_title('Emprunts par région et famille')
axes[0].set_xlabel('nombre d’événements')
axes[1].plot(activity_plot['periode'], activity_plot['emprunts'], label='emprunts')
axes[1].plot(activity_plot['periode'], activity_plot['en_retard'], label='en retard')
axes[1].plot(activity_plot['periode'], activity_plot['perdus'], label='perdus')
axes[1].set_title('Activité mensuelle')
axes[1].tick_params(axis='x', rotation=60)
axes[1].legend()
fig.tight_layout()
plt.show()

spark.stop()

### Limites assumées

1. Le jeu est synthétique : les comportements mesurés ne décrivent pas une population réelle d’usagers.
2. Un statut indique l’état enregistré de l’événement ; il ne permet pas de reconstituer l’historique si plusieurs lignes partagent un id_emprunt.
3. La valeur déclarée d’un ouvrage perdu est une estimation de remplacement, pas une perte comptable auditée.
4. Les emprunts sans ouvrage, site ou adhérent correspondant restent conservés mais ne peuvent pas être attribués à leurs dimensions.
5. Les durées manquantes sont exclues des moyennes ; leurs causes ne peuvent pas être inférées du jeu seul.
6. Spark est mesuré en local sur une seule machine ; le résultat ne prouve pas un avantage de vitesse pour ce volume.

---
## 6. Contrôle avant dépôt

La cellule suivante vérifie que votre rendu est complet. Exécutez-la avant de déposer.

Le dépôt est attendu le **jour 5 à 13 h 30**, avant la reprise de l'après-midi : votre
composant PySpark se termine le matin même. Prévoyez trente minutes pour la mise en forme,
pas cinq.

In [ ]:
checks = {
    'requirements.txt présent': (ROOT / 'requirements.txt').exists(),
    'README.md présent': (ROOT / 'README.md').exists(),
    'dossier src/ non vide': (ROOT / 'src').exists() and any((ROOT / 'src').glob('*.py')),
    'jeu final produit': (any(PROCESSED_DIR.glob('*.parquet'))
                          or any(PROCESSED_DIR.glob('*.csv'))),
    'note méthodologique': any(ROOT.glob('note_methodologique.*')),
    'support de soutenance': any(ROOT.glob('soutenance.*')),
}

width = max(len(label) for label in checks)
for label, is_ok in checks.items():
    print(f"{label:<{width}}  {'OK' if is_ok else 'MANQUANT'}")

if all(checks.values()):
    print('\nRendu complet.')
else:
    print('\nIl manque des éléments listés ci-dessus.')

## 7. Préparation aux questions

1. **Pourquoi laisser les durées manquantes ?** Aucune autre colonne ne permet de les reconstruire. Une moyenne imputée masquerait l’incertitude ; elles restent dans les comptages, mais pas dans la moyenne de durée.
2. **Pourquoi ne pas supprimer toutes les lignes avec un id_emprunt répété ?** Les répétitions peuvent refléter une correction de statut. Je retire seulement les lignes strictement identiques et j’inspecte les identifiants réutilisés avant toute règle métier.
3. **Pourquoi normaliser statut et mode ?** Les espaces et la casse créent de fausses catégories et faussent les taux de retard/perte.
4. **Comment traiter une clé orpheline ?** Je conserve la ligne avec une jointure gauche, compte les non-appariements et évite de lui inventer une région ou une famille.
5. **Pourquoi utiliser le broadcast ?** Les référentiels sont petits. Spark peut les diffuser aux workers, ce qui évite de redistribuer la grande table d’événements.
6. **Que signifie valeur_perdue_eur ?** C’est la valeur déclarée dans la source des événements au statut perdu ; ce n’est pas une mesure comptable auditée.
7. **Quelle est la limite principale ?** Les données synthétiques et les identifiants répétés empêchent de traiter le résultat comme un historique réel sans validation métier.

Chaque membre du groupe doit renseigner son nom et pouvoir expliquer toutes les décisions.